In this code we estimate the state transition equations for $R_t$ and $P_t$ and try and replicate the  the paper’s estimates. The estiamtes from the paper can be seen below:
$$
\begin{array}{lc}
\hline
\textbf{Statistic} & \textbf{Target Value} \\
\hline
\text{Avg. Production Cost }(c) & 231\ (\text{Real2000})  \\
\text{Residual Correlation} & 0.054 \\
\text{Residual Covariance} & 0.000886 \\
\text{Price Splicing Ratio} & 0.868  \\
\ln R_t \text{ Lag Coefficient} & 0.742391 \\
\hline
\end{array}
$$

In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
folder = '/Users/mikkelrathtornerup/Desktop/Dynamic-Programming-Project-main/Data'
filename = 'Lohano_King_Data_Cleaned.csv'

filepath = os.path.join(folder, filename)
df = pd.read_csv(filepath, sep=';')
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns.tolist())

Columns:
['Year', 'Gross Return (nominal)', 'Costs (nominal)', 'Gross Return (nominal).1', 'Costs (nominal).1', 'Unnamed: 5', 'Gross Return (nominal).2', 'Costs (nominal).2', 'Gross Return (real) in 2000 dollars', 'Costs (real) in 2000 dollars', 'P: Raup (nominal)', 'P: Taff (nominal)', 'Ratio', 'P  (nominal)', 'P  (real) in 2000 dollars', 'Implicit Price Deflator of GNP', 'RM (percent) (Real)', '(1+rate of return): (Real)']


In [3]:
df = df.rename(columns={
    'Gross Return (real) in 2000 dollars': 'Rt',
    'Costs (real) in 2000 dollars': 'Ct',
    'P  (real) in 2000 dollars': 'Pt',
    '(1+rate of return): (Real)': 'Mt',
    'P: Raup (nominal)': 'P_raup',
    'P: Taff (nominal)': 'P_taff'
})

In [4]:
# Convert to numeric
for col in ['Year', 'Rt', 'Ct', 'Pt', 'Mt', 'P_raup', 'P_taff']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Sort once by year
df = df.sort_values('Year').reset_index(drop=True)

#### 1. RETURN EQUATION 
$$ ln(R_t) = beta0 + beta1 * ln(R_{t-1}) + error_t $$


In [7]:
# 1. RETURN EQUATION 
df_R = df[['Year', 'Rt']].copy()
df_R = df_R.dropna(subset=['Year', 'Rt'])
df_R = df_R[df_R['Rt'] > 0].copy()
df_R = df_R.sort_values('Year').reset_index(drop=True)

df_R['ln_Rt'] = np.log(df_R['Rt'])
df_R['ln_Rt_lag'] = df_R['ln_Rt'].shift(1)

# Keep only rows where lag exists
df_R_reg = df_R.dropna(subset=['ln_Rt', 'ln_Rt_lag']).copy()

X_R = sm.add_constant(df_R_reg['ln_Rt_lag'])
y_R = df_R_reg['ln_Rt']

model_R = sm.OLS(y_R, X_R).fit()
eps1 = model_R.resid

print("Return equation years used:",
      int(df_R_reg['Year'].min()), "-", int(df_R_reg['Year'].max()))
print("Return equation observations used:", len(df_R_reg))

Return equation years used: 1968 - 2007
Return equation observations used: 40


#### 2. PRICE EQUATION
This equation should be estimated on rows where Rt and Pt exist:
$$ln(P_{t+1}) = alpha0 + alpha1 * ln(P_t) + alpha2 * ln(R_t) + error_{t+1}$$

In [9]:
# 2. PRICE EQUATION
df_P = df[['Year', 'Rt', 'Pt']].copy()
df_P = df_P.dropna(subset=['Year', 'Rt', 'Pt'])
df_P = df_P[(df_P['Rt'] > 0) & (df_P['Pt'] > 0)].copy()
df_P = df_P.sort_values('Year').reset_index(drop=True)

df_P['ln_Rt'] = np.log(df_P['Rt'])
df_P['ln_Pt'] = np.log(df_P['Pt'])
df_P['ln_Pt_next'] = df_P['ln_Pt'].shift(-1)

df_P_reg = df_P.dropna(subset=['ln_Pt_next', 'ln_Pt', 'ln_Rt']).copy()

X_P = sm.add_constant(df_P_reg[['ln_Pt', 'ln_Rt']])
y_P = df_P_reg['ln_Pt_next']

model_P = sm.OLS(y_P, X_P).fit()
eps2 = model_P.resid

print("Price equation years used:",
      int(df_P_reg['Year'].min()), "-", int(df_P_reg['Year'].max()))
print("Price equation observations used:", len(df_P_reg))

Price equation years used: 1967 - 2006
Price equation observations used: 40


#### 3. RESIDUAL COVARIANCE AND CORRELATION


In [10]:
# 3. RESIDUAL COVARIANCE AND CORRELATION
eps_df = pd.DataFrame({
    'eps1': eps1,
    'eps2': eps2
}).dropna()

residual_cov = eps_df['eps1'].cov(eps_df['eps2'])
residual_corr = eps_df['eps1'].corr(eps_df['eps2'])

#### 4. AVERAGE PRODUCTION COST
Paper uses average real production cost + 30 for labor/management. 

Costs are available only from later years, so this is a separate calculation.

In [11]:
# 4. AVERAGE PRODUCTION COST
df_C = df[['Year', 'Ct']].copy()
df_C = df_C.dropna(subset=['Year', 'Ct']).copy()
df_C = df_C[df_C['Ct'] > 0].copy()

avg_cost_data = df_C['Ct'].mean()
c_total = avg_cost_data + 30

print("Cost years used:",
      int(df_C['Year'].min()), "-", int(df_C['Year'].max()))
print("Cost observations used:", len(df_C))

Cost years used: 1983 - 2007
Cost observations used: 25


#### 5. PRICE SPLICING RATIO
Ratio of Taff to Raup over overlap years


In [12]:
df_ratio = df[['Year', 'P_raup', 'P_taff']].copy()
df_ratio = df_ratio.dropna(subset=['P_raup', 'P_taff']).copy()
df_ratio = df_ratio[(df_ratio['P_raup'] > 0) & (df_ratio['P_taff'] > 0)].copy()

df_ratio['splice_ratio'] = df_ratio['P_taff'] / df_ratio['P_raup']
price_splice = df_ratio['splice_ratio'].mean()

print("Splicing overlap years used:",
      int(df_ratio['Year'].min()), "-", int(df_ratio['Year'].max()))
print("Splicing observations used:", len(df_ratio))

Splicing overlap years used: 1990 - 1992
Splicing observations used: 3


#### 6. RESULTS

In [13]:
# 6. RESULTS
print('\n--- Estimated coefficients ---')
print('\nReturn equation:')
print(model_R.params)

print('\nPrice equation:')
print(model_P.params)

print('\n--- Summary statistics ---')
print(f'Average production cost from data: {avg_cost_data:.6f}')
print(f'Total cost parameter c (+30):      {c_total:.6f}')
print(f'Residual correlation:              {residual_corr:.6f}')
print(f'Residual covariance:               {residual_cov:.6f}')
print(f'Price splicing ratio:              {price_splice:.6f}')
print(f'Return lag coefficient:            {model_R.params["ln_Rt_lag"]:.6f}')

# %%
# Optional: full regression tables
print('\n--- Return equation summary ---')
print(model_R.summary())

print('\n--- Price equation summary ---')
print(model_P.summary())


--- Estimated coefficients ---

Return equation:
const        1.511810
ln_Rt_lag    0.742391
dtype: float64

Price equation:
const   -0.140798
ln_Pt    0.884133
ln_Rt    0.173445
dtype: float64

--- Summary statistics ---
Average production cost from data: 201.094000
Total cost parameter c (+30):      231.094000
Residual correlation:              -0.005561
Residual covariance:               -0.000120
Price splicing ratio:              0.867253
Return lag coefficient:            0.742391

--- Return equation summary ---
                            OLS Regression Results                            
Dep. Variable:                  ln_Rt   R-squared:                       0.533
Model:                            OLS   Adj. R-squared:                  0.521
Method:                 Least Squares   F-statistic:                     43.41
Date:                Fri, 13 Mar 2026   Prob (F-statistic):           8.90e-08
Time:                        15:37:43   Log-Likelihood:                 14.276
